<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Pipeline_Quantz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline_Quantz — Universal Export Pipeline

Exports any checkpoint to FP32 ONNX + INT8 QDQ ONNX for STM32 deployment.

**Design:**
- One shared test set (`exports/shared/`) — generated once, reused by every run
- Each checkpoint gets its own export dir (`exports/<name>/`)
- Every file is skipped if it already exists — safe to re-run at any time
- To add a new checkpoint: add one dict to `RUNS` and re-run

**Never change** `SHARED_DIR` — all STM32 CLI commands must use the shared `test_input.npz`.

In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path

!pip -q install onnx onnxruntime onnxruntime-tools
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import VWW_MobileNetV2, VWW_MobileNetV3
from utils.train   import setup_device, evaluate

device = setup_device(seed=41)

In [ ]:
# ── Config — only this cell needs editing when adding new checkpoints ─
SAVE_DIR   = "/content/drive/My Drive/stm32-thesis/checkpoints"
EXPORT_DIR = Path("/content/drive/My Drive/stm32-thesis/exports")

# Shared test set — one directory for ALL runs, never changes
SHARED_DIR = EXPORT_DIR / "shared"

# ── Add new checkpoints here ──────────────────────────────────────────
# arch: 'mobilenetv2' or 'mobilenetv3'
# name: becomes the export subfolder name under exports/
RUNS = [
    {
        "name" : "mobilenetv2_seed_74",
        "arch" : "mobilenetv2",
        "ckpt" : f"{SAVE_DIR}/mobilenetv2_seed_74.pth",
    },
    {
        "name" : "mobilenetv3_seed_74",
        "arch" : "mobilenetv3",
        "ckpt" : f"{SAVE_DIR}/mobilenetv3_seed_74.pth",
    },
    # ── Add more runs below, e.g.: ────────────────────────────────────
    # {
    #     "name" : "mobilenetv2_pruned_unstructured_30pct",
    #     "arch" : "mobilenetv2",
    #     "ckpt" : f"{SAVE_DIR}/mobilenetv2_pruned_unstructured_30pct.pth",
    # },
]

# Create all dirs
SHARED_DIR.mkdir(parents=True, exist_ok=True)
for r in RUNS:
    r["out_dir"] = EXPORT_DIR / r["name"]
    r["out_dir"].mkdir(parents=True, exist_ok=True)

# Pre-flight check
print(f"Shared test dir : {SHARED_DIR}")
print(f"\nRuns ({len(RUNS)} total):")
for r in RUNS:
    status = "✅" if Path(r["ckpt"]).exists() else "❌ MISSING"
    print(f"  {status}  {r['name']:<40}  {r['ckpt']}")

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────

def skip(path: Path, label: str) -> bool:
    """Print a skip message and return True if file already exists."""
    if path.exists():
        print(f"    ⏭️  {label} already exists, skipping")
        return True
    return False


def build_model(arch: str) -> nn.Module:
    if arch == "mobilenetv2":
        return VWW_MobileNetV2().to(device)
    elif arch == "mobilenetv3":
        return VWW_MobileNetV3().to(device)
    raise ValueError(f"Unknown arch: {arch!r}")


def generate_shared_test_files(test_loader, shared_dir: Path, n: int = 200) -> None:
    """
    Generate test_input.npz and test_labels.npz once into shared_dir.
    These files are model-agnostic — inputs and labels only, no logits.
    Skipped automatically if they already exist.
    """
    input_path  = shared_dir / "test_input.npz"
    labels_path = shared_dir / "test_labels.npz"

    if input_path.exists() and labels_path.exists():
        inp = np.load(input_path)["input"]
        lbl = np.load(labels_path)["label"]
        print(f"    ⏭️  Shared test files already exist — {inp.shape}  labels: {lbl.shape}")
        return

    print("    Generating shared test files (model-agnostic inputs + labels)...")
    inputs, labels = [], []
    for i, (x, y) in enumerate(test_loader):
        if i >= n:
            break
        inputs.append(x.numpy().astype("float32")[0])
        labels.append(int(y.item()))
    inputs = np.stack(inputs)
    labels = np.array(labels, dtype="int32")
    np.savez(input_path,  input=inputs)
    np.savez(labels_path, label=labels)
    print(f"    ✅ Shared test files saved — {inputs.shape}  labels: {labels.shape}")


def export_onnx(model: nn.Module, path: Path) -> None:
    if skip(path, "FP32 ONNX"): return
    model.eval()
    dummy = torch.randn(1, 3, 96, 96, device=device)
    torch.onnx.export(
        model, dummy, str(path),
        input_names=["input"], output_names=["logits"],
        export_params=True, opset_version=18,
        do_constant_folding=True,
        dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
        dynamo=False,
    )
    onnx.checker.check_model(str(path), full_check=False)
    print(f"    ✅ FP32 ONNX: {path}")


def save_calib_file(train_loader, path: Path, n: int = 200) -> None:
    if skip(path, "Calib data"): return
    xs = []
    with torch.no_grad():
        for i, (x, _) in enumerate(train_loader):
            if i >= n: break
            xs.append(x.numpy().astype("float32")[0])
    xs = np.stack(xs)
    np.savez(path, input=xs)
    print(f"    ✅ Calib data saved: {path}  {xs.shape}")


class CalibReader(CalibrationDataReader):
    def __init__(self, npz_path):
        self.data = np.load(npz_path)["input"].astype("float32")
        self.i = 0
    def get_next(self):
        if self.i >= len(self.data): return None
        out = {"input": self.data[self.i : self.i + 1]}
        self.i += 1
        return out
    def rewind(self): self.i = 0


def quantize_int8(fp32_path: Path, calib_path: Path, int8_path: Path) -> None:
    if skip(int8_path, "INT8 ONNX"): return
    quantize_static(
        model_input=str(fp32_path),
        model_output=str(int8_path),
        calibration_data_reader=CalibReader(calib_path),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
    )
    print(f"    ✅ INT8 QDQ ONNX: {int8_path}")


def onnx_sanity_check(fp32_path: Path, input_npz: Path) -> None:
    sess   = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
    sample = np.load(input_npz)["input"][0:1]
    out    = sess.run(["logits"], {"input": sample})[0]
    print(f"    ✅ ONNX sanity — shape: {out.shape}  logits: {out[0]}")


def compute_stm32_accuracy(
    labels_npz: Path,
    outputs_npz: Path,
    key: str = "c_outputs_1",
    num_classes: int = 2,
) -> float:
    labels = np.load(labels_npz)["label"].astype("int64")
    raw    = np.load(outputs_npz)[key]
    if raw.size != len(labels) * num_classes:
        raise ValueError(f"Size mismatch: {raw.size} vs {len(labels) * num_classes}")
    preds = np.argmax(raw.reshape(len(labels), num_classes), axis=1)
    return float((preds == labels).mean() * 100)

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64)
test_loader              = get_test_loader(batch_size=1)
print("✅ Dataset ready")

In [ ]:
# ── Shared test files — generated once, reused by all runs ───────────
# No model involved — these are raw inputs + ground-truth labels only.
# If the files already exist this cell is a no-op.
print("[Shared test set]")
generate_shared_test_files(test_loader, SHARED_DIR, n=200)

SHARED_INPUT_NPZ  = SHARED_DIR / "test_input.npz"
SHARED_LABELS_NPZ = SHARED_DIR / "test_labels.npz"

In [ ]:
# ── Main export loop ─────────────────────────────────────────────────
# Skips any file that already exists — safe to re-run.
# New runs added to RUNS will be processed; completed ones are skipped.
results = []

for run in RUNS:
    name    = run["name"]
    arch    = run["arch"]
    ckpt    = run["ckpt"]
    out_dir = run["out_dir"]

    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    if not Path(ckpt).exists():
        print(f"  ❌ Checkpoint missing, skipping: {ckpt}")
        continue

    # [1] Load model
    print("[1/4] Loading model...")
    model = build_model(arch)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()
    val_acc = evaluate(model, val_loader, device)
    print(f"    Val accuracy: {val_acc * 100:.2f}%")

    # [2] FP32 ONNX
    print("[2/4] FP32 ONNX...")
    fp32_path = out_dir / "model_fp32.onnx"
    export_onnx(model, fp32_path)

    # [3] Calibration data (train split only)
    print("[3/4] Calibration data...")
    calib_npz = out_dir / "calib_train.npz"
    save_calib_file(train_loader, calib_npz, n=200)

    # [4] INT8 QDQ
    print("[4/4] INT8 QDQ quantisation...")
    int8_path = out_dir / "model_int8_qdq.onnx"
    quantize_int8(fp32_path, calib_npz, int8_path)

    # Sanity check against shared inputs
    onnx_sanity_check(fp32_path, SHARED_INPUT_NPZ)

    results.append({
        "name"      : name,
        "arch"      : arch,
        "val_acc"   : val_acc * 100,
        "fp32_path" : fp32_path,
        "int8_path" : int8_path,
        "out_dir"   : out_dir,
    })
    print(f"  ✅ {name} done.")

print(f"\n{'='*60}")
print(f"  Done. {len(results)}/{len(RUNS)} runs exported.")
print(f"  Shared test set : {SHARED_DIR}")
print(f"{'='*60}")

## STM32 CLI Commands

All models use the **same** `test_input.npz` from `exports/shared/`.

```bash
# Template — replace <name> with the run name (e.g. mobilenetv2_seed_74)

# FP32
stedgeai validate -m exports/<name>/model_fp32.onnx \
                  --target stm32n6 --mode target \
                  --valinput exports/shared/test_input.npz
cp network_val_io.npz exports/<name>/output_fp32.npz

# INT8
stedgeai validate -m exports/<name>/model_int8_qdq.onnx \
                  --target stm32n6 --mode target \
                  --valinput exports/shared/test_input.npz
cp network_val_io.npz exports/<name>/output_int8.npz
```

In [ ]:
# ── STM32 on-device accuracy ─────────────────────────────────────────
# Run after STM32 CLI outputs are copied back.
# Shows 'pending' for any run not yet completed.

def _acc_str(out_dir, fname):
    p = Path(out_dir) / fname
    if not p.exists():
        return "pending", float("nan")
    try:
        return f"{compute_stm32_accuracy(SHARED_LABELS_NPZ, p):.2f}%", compute_stm32_accuracy(SHARED_LABELS_NPZ, p)
    except Exception as e:
        return f"error", float("nan")

W = 42
print(f"{'Run':<{W}} {'Val Acc':>8} {'FP32':>10} {'INT8':>10} {'Delta':>8}")
print("-" * (W + 40))

for r in results:
    fp32_str, fp32_acc = _acc_str(r["out_dir"], "output_fp32.npz")
    int8_str, int8_acc = _acc_str(r["out_dir"], "output_int8.npz")
    delta = f"{int8_acc - fp32_acc:+.2f}%" if (int8_acc == int8_acc and fp32_acc == fp32_acc) else "—"
    print(f"{r['name']:<{W}} {r['val_acc']:>7.2f}% {fp32_str:>10} {int8_str:>10} {delta:>8}")